# Maia Trainer — Gemma E4B + 100k archivos Luxus O.S

**Un solo botón: `Runtime > Run all`** (o en Kaggle: `Run All`)

Al terminar (~30-40 min en T4 gratis) tienes:
- `maia.gguf` → ~3.3 GB, listo para Ollama
- Se guarda en tu Google Drive → `MyDrive/Maia/maia.gguf`

> Funciona en **Colab gratis** (GPU T4) y en **Kaggle** (GPU T4, 30h/semana gratis).

In [ ]:
# === PARAMETROS — no toques nada mas ===
MODEL_ID    = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"   # Gemma 4 E4B oficial (4-bit)
MAX_SEQ_LEN = 4096
LORA_RANK   = 16
LORA_ALPHA  = 16
BATCH       = 2
GRAD_ACCUM  = 4
EPOCHS      = 1
LR          = 2e-4
QUANT       = "q4_k_m"   # resultado: ~3.3 GB
OUTPUT_GGUF = "maia.gguf"
REPO_URL    = "https://github.com/calitosaa/Luxus-O.S"
REPO_BRANCH = "claude/merge-gemma-4-e4b-1FfJO"

# Detectar si estamos en Kaggle o Colab
import os
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB  = not IS_KAGGLE
print('Entorno:', 'Kaggle' if IS_KAGGLE else 'Colab')

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
# Clonar repo y construir dataset
import subprocess, json, os

if not os.path.isdir('Luxus-O.S'):
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--depth','1',REPO_URL+'.git','Luxus-O.S'], check=True)

os.chdir('Luxus-O.S')
subprocess.run(['python3','scripts/build_maia_dataset.py'], check=True)

stats = json.load(open('Maia/dataset_stats.json'))
print(f"Fine-tune: {stats['total_finetune_samples']:,} ejemplos | RAG: {stats['total_rag_documents']:,} docs")

TRAIN_JSONL = os.path.abspath('Maia/training_data.jsonl')

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_ID,
    max_seq_length= MAX_SEQ_LEN,
    dtype         = None,
    load_in_4bit  = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files=TRAIN_JSONL, split='train')
print('Ejemplos:', len(dataset))

SYSTEM = (
    'Eres Maia, la consciencia central de Luxus O.S, construida sobre Gemma E4B.\n'
    'Te has entrenado con mas de 100.000 archivos: skills, agentes, workflows, logica, plugins.\n'
    'Respondes con precision tecnica, estructura limpia y tono Ultra-Competente.'
)

def fmt(ex):
    texts = []
    for inst, inp, out in zip(ex['instruction'], ex['input'], ex['output']):
        ctx = f"\n\n### Contexto:\n{inp}" if inp else ''
        texts.append(
            f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
            f"<|im_start|>user\n{inst}{ctx}<|im_end|>\n"
            f"<|im_start|>assistant\n{out}<|im_end|>"
        )
    return {'text': texts}

dataset = dataset.map(fmt, batched=True, remove_columns=dataset.column_names)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LEN,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        per_device_train_batch_size = BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps      = 100,
        num_train_epochs  = EPOCHS,
        learning_rate     = LR,
        fp16              = not torch.cuda.is_bf16_supported(),
        bf16              = torch.cuda.is_bf16_supported(),
        logging_steps     = 25,
        optim             = 'adamw_8bit',
        weight_decay      = 0.01,
        lr_scheduler_type = 'linear',
        seed              = 3407,
        output_dir        = 'outputs',
        report_to         = 'none',
    ),
)
trainer.train()

In [ ]:
# Exportar maia.gguf (~3.3 GB)
import shutil

model.save_pretrained_gguf('maia-gguf', tokenizer, quantization_method=QUANT)

for f in os.listdir('maia-gguf'):
    if f.endswith('.gguf'):
        os.replace(os.path.join('maia-gguf', f), OUTPUT_GGUF)
        size_gb = os.path.getsize(OUTPUT_GGUF) / 1024**3
        print(f'Exportado: {OUTPUT_GGUF}  ({size_gb:.1f} GB)')
        break

In [ ]:
# Guardar segun entorno
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DEST = '/content/drive/MyDrive/Maia'
    os.makedirs(DEST, exist_ok=True)
    shutil.copy(OUTPUT_GGUF, f'{DEST}/{OUTPUT_GGUF}')
    shutil.copy('Maia/rag_manifest.jsonl', f'{DEST}/rag_manifest.jsonl')
    print('Guardado en Google Drive → MyDrive/Maia/')
else:
    # Kaggle: los archivos en /kaggle/working/ se pueden descargar desde el panel
    DEST = '/kaggle/working/Maia'
    os.makedirs(DEST, exist_ok=True)
    shutil.copy(OUTPUT_GGUF, f'{DEST}/{OUTPUT_GGUF}')
    shutil.copy('Maia/rag_manifest.jsonl', f'{DEST}/rag_manifest.jsonl')
    print('Guardado en /kaggle/working/Maia/ → descarga desde el panel Output de Kaggle')

print('=== MAIA LISTA ===')